# 新機能

- [whats news](https://aws.amazon.com/jp/about-aws/whats-new/2025/07/amazon-nova-canvas-virtual-try-on-style-options-image-generation/)
- [blog](https://aws.amazon.com/jp/blogs/aws/amazon-nova-canvas-update-virtual-try-on-and-style-options-now-available/)
- [blog (jp)](https://aws.amazon.com/jp/blogs/news/amazon-nova-canvas-update-virtual-try-on-and-style-options-now-available/)
- [doc-param](https://docs.aws.amazon.com/nova/latest/userguide/image-gen-req-resp-structure.html)
- [classmethod](https://dev.classmethod.jp/articles/amazon-nova-canvas-virtual-try-on-style-options-image-generation/)
- [aws-sample](https://github.com/aws-samples/sample-genai-design-studio/tree/main)


## helper


In [49]:
import base64
import io
import json

import boto3
from PIL import Image


def generate_image(
    payload: dict,
    num_image: int = 1,
    cfg_scale: float = 6.5,
    seed: int = 42,
    model_id: str = "amazon.nova-canvas-v1:0",
) -> None:
    client = boto3.client("bedrock-runtime", region_name="us-east-1")
    body = json.dumps(
        {
            **payload,
            "imageGenerationConfig": {
                "numberOfImages": num_image,  # Range: 1 to 5
                "quality": "premium",  # Options: standard/premium
                "height": 1024,  # Supported height list above
                "width": 1024,  # Supported width list above
                "cfgScale": cfg_scale,  # Range: 1.0 (exclusive) to 10.0
                "seed": seed,  # Range: 0 to 214783647
            },
        }
    )

    response = client.invoke_model(
        body=body,
        modelId=model_id,
        accept="application/json",
        contentType="application/json",
    )

    response_body = json.loads(response.get("body").read())
    base64_image = response_body.get("images")[0]
    base64_bytes = base64_image.encode("ascii")
    image_bytes = base64.b64decode(base64_bytes)

    image = Image.open(io.BytesIO(image_bytes))
    image.show()

    # save the image
    image.save(f"../images/virtual_try_on/generated_image_seed={seed}.png")

In [50]:
def load_image_as_base64(image_path) -> str:
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [51]:
human_image_path = "../images/virtual_try_on/human.png"
shirt_image_path = "../images/virtual_try_on/swag_aws_cb_cap.jpg"
resized_shirt_image_path = "../images/virtual_try_on/swag_aws_cb_cap_resized.jpg"

In [53]:
# check image size
img = Image.open(shirt_image_path)
print(f"Image size: {img.size[0]}x{img.size[1]} pixels")

Image size: 4284x4284 pixels


In [54]:
# resize image to 2048x2048 pixels
img = img.resize((2048, 2048), Image.LANCZOS)
img.save(resized_shirt_image_path)

In [ ]:
generate_image(
    {
        "taskType": "VIRTUAL_TRY_ON",
        "virtualTryOnParams": {
            "sourceImage": load_image_as_base64(human_image_path),
            "referenceImage": load_image_as_base64(resized_shirt_image_path),
            "maskType": "GARMENT",
            "garmentBasedMask": {"garmentClass": "UPPER_BODY"},
        },
    },
)

In [ ]:
generate_image(
    {
        "taskType": "VIRTUAL_TRY_ON",
        "virtualTryOnParams": {
            "sourceImage": load_image_as_base64(human_image_path),
            "referenceImage": load_image_as_base64(resized_shirt_image_path),
            "maskType": "PROMPT",
            "promptBasedMask": {
                "maskShape": "BOUNDING_BOX",
                "maskPrompt": "head",
            },
        },
    },
)

In [61]:
generate_image(
    {
        "taskType": "VIRTUAL_TRY_ON",
        "virtualTryOnParams": {
            "sourceImage": load_image_as_base64(human_image_path),
            "referenceImage": load_image_as_base64(resized_shirt_image_path),
            "maskType": "PROMPT",
            "promptBasedMask": {
                "maskShape": "BOUNDING_BOX",
                "maskPrompt": "cap",
            },
        },
    },
    seed=12345,  # Change seed for different results
)

ValidationException: An error occurred (ValidationException) when calling the InvokeModel operation: Model produced invalid mask with provided [maskPrompt]. Please refer to the model documentation and update the input before trying again.